<a href="https://colab.research.google.com/github/mnijhuis-dnb/AI_methods_in_economics_and_finance_research/blob/main/exercises/exercise_2_pytorch_basic_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import gdown
from pathlib import Path
import os

# Install gdown if not already present
# !pip install gdown

# Define the Google Drive file ID from the provided URL
file_id = "1--l9cVrHUtivNwktbvGXyntzAlXfS6_s"

# Define the target directory and filename for the compressed data
# The notebook's DATA_PATH (defined in cell jXS26JMMJfpn) expects
# the uncompressed file at '../data/census-income.data'.
# The _unzip_data function expects the compressed file at DATA_PATH + ".bz2".
# So, the compressed file should be saved as '../data/census-income.data.bz2'.
data_dir = Path("../data")
data_dir.mkdir(parents=True, exist_ok=True) # Ensure the directory exists

output_filepath = data_dir / "census-income.data"

print(f"Downloading data from Google Drive (ID: {file_id}) to {output_filepath}...")

# Check if the file already exists and is not empty to avoid re-downloading
if output_filepath.exists() and output_filepath.stat().st_size > 0:
    print(f"File '{output_filepath}' already exists. Skipping download.")
else:
    try:
        # Download the file using gdown
        gdown.download(id=file_id, output=str(output_filepath), quiet=False)
        print("Download complete.")
    except Exception as e:
        print(f"Error downloading file: {e}")
        print("Please ensure 'gdown' is installed (`!pip install gdown`) or check the file ID and permissions.")
from pathlib import Path
import bz2
import shutil

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

DATA_PATH = "../data/census-income.data"

DATA_TYPES = [
    ("age", "float64"),
    ("class of worker", "category"),
    ("detailed industry recode", "category"),
    ("detailed occupation recode", "category"),
    ("education", "category"),
    ("wage per hour", "float64"),
    ("enroll in edu inst last wk", "category"),
    ("marital stat", "category"),
    ("major industry code", "category"),
    ("major occupation code", "category"),
    ("race", "category"),
    ("hispanic origin", "category"),
    ("sex", "category"),
    ("member of a labor union", "category"),
    ("reason for unemployment", "category"),
    ("full or part time employment stat", "category"),
    ("capital gains", "float64"),
    ("capital losses", "float64"),
    ("dividends from stocks", "float64"),
    ("tax filer stat", "category"),
    ("region of previous residence", "category"),
    ("state of previous residence", "category"),
    ("detailed household and family stat", "category"),
    ("detailed household summary in household", "category"),
    ("instance weight_ignore", "float64"),
    ("migration code-change in msa", "category"),
    ("migration code-change in reg", "category"),
    ("migration code-move within reg", "category"),
    ("live in this house 1 year ago", "category"),
    ("migration prev res in sunbelt", "category"),
    ("num persons worked for employer", "float64"),
    ("family members under 18", "category"),
    ("country of birth father", "category"),
    ("country of birth mother", "category"),
    ("country of birth self", "category"),
    ("citizenship", "category"),
    ("own business or self employed", "category"),
    ("fill inc questionnaire for veteran's admin", "category"),
    ("veterans benefits", "category"),
    ("weeks worked in year", "float64"),
    ("year", "category"),
    ("targets", "category"),
]

EDU_CODE = {
    "Children": 0,
    "Less than 1st grade": 1,
    "1st 2nd 3rd or 4th grade": 2,
    "5th or 6th grade": 3,
    "7th and 8th grade": 4,
    "9th grade": 5,
    "10th grade": 6,
    "11th grade": 7,
    "12th grade no diploma": 8,
    "High school graduate": 9,
    "Some college but no degree": 10,
    "Associates degree-academic program": 11,
    "Associates degree-occup /vocational": 12,
    "Bachelors degree(BA AB BS)": 13,
    "Masters degree(MA MS MEng MEd MSW MBA)": 14,
    "Prof school degree (MD DDS DVM LLB JD)": 15,
    "Doctorate degree(PhD EdD)": 15,
}

BINARY_COLUMNS = [
    "member of a labor union",
    "live in this house 1 year ago",
    "own business or self employed",
    "fill inc questionnaire for veteran's admin",
    "veterans benefits",
    "year",
    "sex",
]

def load_raw_data(path):
    """Load the raw census income dataset into a pandas DataFrame.

    Parameters
    ----------
    path : str or pathlib.Path or None
        Path to the uncompressed dataset file. When ``None``, the default
        module path is used and the compressed file is extracted if needed.

    Returns
    -------
    pandas.DataFrame
        Raw census data with column names and dtypes defined by ``DATA_TYPES``.
    """
    if path is None:
        path = DATA_PATH
        if not Path(path).exists():
            _unzip_data()

    return pd.read_csv(path, names=[d[0] for d in DATA_TYPES], dtype=dict(DATA_TYPES))

def _unzip_data():
    """Extract the bundled compressed census income dataset.

    Returns
    -------
    None
        The function writes the extracted file next to the compressed archive.
    """

    source_path = Path(DATA_PATH + ".bz2")
    target_path = source_path.with_suffix("")

    with bz2.open(source_path, "rb") as src, target_path.open("wb") as dst:
        shutil.copyfileobj(src, dst)


def add_education_num(raw_data):
    """Add a numeric encoding of the education label column.

    Parameters
    ----------
    raw_data : pandas.DataFrame
        Input census dataset containing the ``education`` column.

    Returns
    -------
    pandas.DataFrame
        Copy of ``raw_data`` with an additional ``education-num`` column.
    """
    raw_data = raw_data.copy()
    raw_data["education-num"] = np.array(
        [EDU_CODE[v.strip()] for v in raw_data["education"]]
    ).astype("float64")
    return raw_data


def build_features(raw_data):
    """Transform raw census records into a model-ready feature matrix.

    Parameters
    ----------
    raw_data : pandas.DataFrame
        Census records without the target column.

    Returns
    -------
    pandas.DataFrame
        Feature matrix containing one-hot encoded categorical features,
        numerical columns, and factorized binary columns.
    """
    raw_data = add_education_num(raw_data)
    data = raw_data.drop(
        columns=[
            "instance weight_ignore",
            "detailed industry recode",
            "detailed occupation recode",
            "major industry code",
            "major occupation code",
            "country of birth father",
            "country of birth mother",
            "country of birth self",
            "state of previous residence",
            "detailed household and family stat",
            "education",
        ]
    )

    binary_data = data[BINARY_COLUMNS].copy()
    categorical_data = data.select_dtypes(include=["category"]).drop(
        columns=BINARY_COLUMNS, errors="ignore"
    )
    numerical_data = data.select_dtypes(include=["float64", "int64"]).drop(
        columns=BINARY_COLUMNS, errors="ignore"
    )

    binary_data[
        (binary_data == " 2")
        | (binary_data == " ?")
        | (binary_data == " Not in universe")
        | (binary_data == " Not in universe under 1 year old")
    ] = np.nan
    binary_data = binary_data.apply(lambda x: pd.factorize(x)[0])

    encoder = OneHotEncoder(handle_unknown="ignore")
    encoder.fit(categorical_data)

    categorical_source = data.select_dtypes(include=["category"]).drop(
        columns=BINARY_COLUMNS, errors="ignore"
    )

    return pd.concat(
        [
            pd.DataFrame(
                encoder.transform(categorical_source).toarray(),
                index=categorical_source.index,
                columns=encoder.get_feature_names_out(categorical_source.columns),
            ),
            numerical_data,
            binary_data,
        ],
        axis=1,
    )


def rebalance_training_data(train_x, train_y, random_state=99):
    """Downsample the majority class to create a balanced training set.

    Parameters
    ----------
    train_x : pandas.DataFrame
        Training features.
    train_y : pandas.Series
        Binary target labels aligned with ``train_x``.
    random_state : int, default=99
        Seed used for deterministic resampling.

    Returns
    -------
    tuple[pandas.DataFrame, pandas.Series]
        Resampled training features and labels with balanced classes.
    """
    rng = np.random.default_rng(random_state)
    positive_idx = train_y.index.values[train_y == 1]
    negative_idx = rng.permutation(train_y.index.values[train_y == 0])
    index = rng.permutation(np.hstack((positive_idx, negative_idx))[: 2 * len(positive_idx)])
    return train_x.loc[index], train_y.loc[index]


def prepare_census_data(
    path=None,
    random_state=99,
    split_validation=False,
    rebalance=True,
):
    """Load, featurize, split, and optionally rebalance the census dataset.

    Parameters
    ----------
    path : str or pathlib.Path or None, default=None
        Path to the raw census data file. When ``None``, the default dataset
        path is used.
    random_state : int, default=99
        Seed used for train-test splitting and optional rebalancing.
    split_validation : bool, default=False
        Whether to create a separate validation split in addition to the test
        split.
    rebalance : bool, default=True
        Whether to rebalance the training data after splitting.

    Returns
    -------
    tuple
        Either ``(train_x, test_x, train_y, test_y)`` or
        ``(train_x, val_x, test_x, train_y, val_y, test_y)`` depending on
        ``split_validation``.
    """
    raw_data = load_raw_data(path)
    targets, _ = pd.factorize(raw_data["targets"])
    features = build_features(raw_data.drop(columns=["targets"]))
    target_series = pd.Series(targets, index=features.index)

    if split_validation:
        train_x, temp_x, train_y, temp_y = train_test_split(
            features, target_series, random_state=random_state
        )
        val_x, test_x, val_y, test_y = train_test_split(
            temp_x, temp_y, random_state=random_state
        )
        if rebalance:
            train_x, train_y = rebalance_training_data(train_x, train_y, random_state)
        return train_x, val_x, test_x, train_y, val_y, test_y

    train_x, test_x, train_y, test_y = train_test_split(
        features, target_series, random_state=random_state
    )
    if rebalance:
        train_x, train_y = rebalance_training_data(train_x, train_y, random_state)
    return train_x, test_x, train_y, test_y

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
train_x, val_x, test_x, train_y, val_y, test_y = prepare_census_data(
    random_state=99,
    split_validation=True,
    rebalance=True,
 )

train_x.head()

Get the data in tensor format

In [ ]:
train_x_tensor = torch.tensor(train_x.to_numpy(), dtype=torch.float32)
val_x_tensor = torch.tensor(val_x.to_numpy(), dtype=torch.float32)
test_x_tensor = torch.tensor(test_x.to_numpy(), dtype=torch.float32)
train_y_tensor = torch.tensor(train_y.to_numpy(), dtype=torch.float32).unsqueeze(1)
val_y_tensor = torch.tensor(val_y.to_numpy(), dtype=torch.float32).unsqueeze(1)
test_y_tensor = torch.tensor(test_y.to_numpy(), dtype=torch.float32).unsqueeze(1)

Normalize the data

In [ ]:
train_mean = train_x_tensor.mean(dim=0, keepdim=True)
train_std = train_x_tensor.std(dim=0, keepdim=True)
train_std = torch.where(train_std == 0, torch.ones_like(train_std), train_std)

train_x_tensor = (train_x_tensor - train_mean) / train_std
val_x_tensor = (val_x_tensor - train_mean) / train_std
test_x_tensor = (test_x_tensor - train_mean) / train_std

Make the dataloaders

In [ ]:
train_loader = DataLoader(
    TensorDataset(train_x_tensor, train_y_tensor),
    batch_size=512,
    shuffle=True,
 )
val_loader = DataLoader(TensorDataset(val_x_tensor, val_y_tensor), batch_size=1024)
test_loader = DataLoader(TensorDataset(test_x_tensor, test_y_tensor), batch_size=1024)

Define a simple network

In [ ]:
class BasicNN(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()

        self.input_size = input_size
        self.num_classes = num_classes

        self.linear_1 = nn.Linear(input_size, 5)
        self.linear_2 = nn.Linear(5, num_classes)

    def forward(self, x):
        x = nn.ReLU()(self.linear_1(x))
        x = nn.Sigmoid()(self.linear_2(x))

        return x

Inspect the model

After defining a model, it is useful to print its layers, inspect parameter shapes, and count how many trainable parameters it has.

In [ ]:
inspection_model = BasicNN(input_size=train_x_tensor.shape[1], num_classes=1)
print(inspection_model)

for name, parameter in inspection_model.named_parameters():
    print(f"{name}: shape={tuple(parameter.shape)}, trainable={parameter.requires_grad}")

sum(parameter.numel() for parameter in inspection_model.parameters() if parameter.requires_grad)

Define an evaluation function

In [ ]:
def evaluate_model(model, data_loader, device, threshold=0.5):
    model.eval()
    all_targets = []
    all_predictions = []

    with torch.no_grad():
        for features, targets in data_loader:
            features = features.to(device)
            predictions = model(features)
            all_predictions.append(predictions)
            all_targets.append(targets)

    y_true = torch.cat(all_targets).squeeze(1).numpy()
    y_pred = torch.cat(all_predictions).squeeze(1).numpy()
    y_pred = (y_pred >= threshold).astype(int)

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

Define the training function

In [ ]:
def train_model(model, train_loader, val_loader, device, epochs=12, learning_rate=1e-3):
    torch.manual_seed(99)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.BCEWithLogitsLoss()

    history = []
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for features, targets in train_loader:
            features = features.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            logits = model(features)
            loss = loss_fn(logits, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * features.size(0)

        validation_metrics = evaluate_model(model, val_loader, device)
        history.append(
            {
                "epoch": epoch + 1,
                "train_loss": running_loss / len(train_loader.dataset),
                **validation_metrics,
            }
        )

    return model, pd.DataFrame(history)

Train the model

In [ ]:
model = BasicNN(
        input_size=126,
        num_classes=1
    ).to(device)
trained_model, training_history = train_model(
    model, train_loader, val_loader, device, epochs=10, learning_rate=2e-3
)

In [ ]:
ax = training_history.plot(x="epoch", y="train_loss", figsize=(12, 8), color="tab:blue")
training_history.plot(
    x="epoch",
    y="recall",
    secondary_y=True,
    ax=ax,
    color="tab:orange",
)

ax.set_ylabel("train_loss")
ax.right_ax.set_ylabel("recall");

# Exercise: Discover Neural Network Architectures

We have a very basic neural network to determine if someone is earning more than $50k, but we can improve the performance of the basic model by change the architecture of the model. Your goal is to improve the predictive performance of this model by experimenting with the network architecture. You are free to modify the model, but keep the data used in the model the same.

### What you can change
Try different architectural choices such as:
- Depth; Adding or removing layers
- Width; Increase or decrease the number of neurons per layer
- Activation functions; Replace ReLU with alternatives (e.g., LeakyReLU, Tanh)
- Regularization: Add dropout layers, use batch normalization
- Output layer behavior; How do we convert the result to 0, 1

When adjusting the network, always also note down how many parameters the network has. Train and evaluate on the same dataset split. As the dataset is very unbalanced we are most interested in having a good recall. Track your results and identify which architectural choice led to improvements.

### Questions to answer
- Which architectural changes had the largest positive impact on performance?
- Did increasing model complexity always improve performance?
- Did you observe signs of overfitting? How did you address them?
- How does your improved model compare to:
- - The baseline model
- - The 10-feature model from the XAI exercise

### Optional bonus (advanced)

- Perform a systematic search (e.g., small grid search over depth/width)
- Compare with a tree-based model (e.g., XGBoost) from the previous exercises
- Reflect on whether the more complex neural network is actually worth it